In [ ]:
# 파이토치 라이브러리 호출
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/rickiepark/hg-mldl/master/wine.csv'
df = pd.read_csv( url ) 

X = df[ ['alcohol', 'sugar', 'pH'] ] #
Y = df['class']           # (0:레드와인, 1:화이트와인)

from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y) 

from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train_scaled = ss.fit_transform( X_train )  # (5197, 3) (5197,)
X_test_scaled = ss.transform( X_test )        # (1300, 3) (1300,)

# 딥러닝을 하기 위한 파이토치( 2의 제곱이 좋음. )
print(X_train_scaled.shape , Y_train.shape)
print(X_test_scaled.shape , Y_test.shape)
# torch.tensor ( 자료 , dtype=torch.타입비트수) : 텐서타입으로 변환
import torch
X_train = torch.tensor(X_train_scaled , dtype=torch.float32)
X_test = torch.tensor(X_test_scaled , dtype=torch.float32)

# 특성과 타겟의 차원수를 일치화
Y_train = torch.tensor(Y_train.values , dtype=torch.float32).unsqueeze(1)
Y_test = torch.tensor(Y_test.values , dtype=torch.float32).unsqueeze(1)

# 인공신경망 ANN 
import torch.nn as nn
# nn.Sequential( 입력층 -> 은닉층 -> 출력층 )
model = nn.Sequential(
    # 입력수
    # nn.Linear(피처수 , 파생피처수)
    nn.Linear(3,16),
    # 활성화함수
    # 음수값는 0으로 양수값은 유지
    nn.ReLU(),
    # 출력층
    # nn.Linear( 앞전 레이어의 피처수 , 결과 수 )
    nn.Linear(16,1)
)

# 손실함수
# 시그모이드 + cross Entropy 하나로 합친
import torch.optim as optim
optimizer = optim.Adam(model.parameters() , lr=0.01)
crierion = nn.BCEWithLogitsLoss()

# 에포크(학습수) 단위 학습(순전파 -> 역전파)
for epoch in range( 10000 ) :
    model.train()
    # 이전 에포크의 계산된 기울기 초기화 
    optimizer.zero_grad()
    logits = model(X_train)
    # 손실(정답과 비교하여 손실 계산)
    loss = crierion(logits , Y_train)
    # (역전파) 각 가중치에 기울기 계산
    loss.backward()
    # (업데이트 반영) 가증치 반영
    optimizer.step()
    # ========== 확인용 ============
    if epoch % 100 == 0:
        print(f'에포크 : {epoch} , 손실(오차) : {loss.item()}')


# 모델 평가
# model.train() : 학습모드
# model.eval()  : 평가/추론모드
# 추론/평가 모드로 전환 
model.eval()
# 기울기 저장 안 함
with torch.no_grad() :
    # 이진분류모델
    # 0.5 보다 크면 1 , 아니면 0 
    train_pred = (torch.sigmoid(model(X_train)) >= 0.5).float()
    test_pred = (torch.sigmoid(model(X_test)) >= 0.5).float()  

# f1-score
from sklearn.metrics import f1_score
print(f1_score(Y_train.numpy() , train_pred , average='macro'))
print(f1_score(Y_test.numpy() , test_pred , average='macro'))


In [ ]:
# 신경망 DNN 이용한 이진분류

import pandas as pd 
url = 'https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-mat.csv' 
df = pd.read_csv( url, sep=';' ) 
X = df[ ['studytime', 'absences', 'G1', 'G2'] ] 
Y = df['G3']                  

from sklearn.model_selection import train_test_split 
X_train, X_test, Y_train, Y_test = train_test_split( X, Y, test_size=0.2, random_state=42 )

from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train_scaled = ss.fit_transform( X_train ) 
X_test_scaled = ss.transform( X_test )   

# 특성과 타겟의 차원수를 일치화
X_train = torch.tensor(X_train_scaled , dtype=torch.float32)
X_test = torch.tensor(X_test_scaled , dtype=torch.float32)
Y_train = torch.tensor(Y_train.values , dtype=torch.float32).unsqueeze(1)
Y_test = torch.tensor(Y_test.values , dtype=torch.float32).unsqueeze(1) 

# 심층신경망 DNN 레이어 컨테이너 설정 
# 입력층 -> 은닉층 - 렐루 - 드롭아웃 -> ... -> 은닉층 - 렐루 - 드롭아웃 -> 은닉층 -> 출력층 )
import torch.nn as nn
model = nn.Sequential(
    # 학습할 특성의 수로 입력층 시작
    # 파생피처 : 아무거나
    nn.Linear(4, 32),
    # 활성화함수
    nn.ReLU(),
    # 매 순전파마다 은닉층에 20% 무작위 비활성화
    nn.Dropout(p=0.2),
    # 앞전 레이어 32노드수 따라서 다음 노드수 설정
    nn.Linear(32, 16),
    # 활성화함수
    nn.ReLU(),
    # 매 순전파마다 은닉층에 20% 무작위 비활성화
    nn.Dropout(p=0.2),
    # 앞전 레이어 16노드수 따라서 다음 노드수 설정
    nn.Linear(16, 8),
    # 활성화함수
    nn.ReLU(),
    # 앞전 레이어 8노드수 따라서 다음 노드수 설정
    nn.Linear(8, 1)
)

# 손실함수 : MSELoss()
import torch.optim as optim
crierion = nn.MSELoss()

# 옵티마이저
optimizer = optim.Adam(model.parameters() , lr=0.01)

# 에포크(학습수) 단위 학습(순전파 -> 역전파)
for epoch in range( 20000 ) :
    # 필수 : 학습모델 실행
    model.train()
    # 이전 에포크의 계산된 기울기 0으로 초기화 
    optimizer.zero_grad()
    # 순전파 / 학습
    logits = model(X_train)
    # 손실 계산 ( 예측값 , 정답지 )
    loss = crierion(logits , Y_train)
    # 역전파
    # 오차 이용하여 가중치와 편향의 기울기/미분 계산
    loss.backward()
    # 업데이트 / 변경된 파라미터(가중치/편향) 반영
    optimizer.step()
    if epoch % 100 == 0:
        print(f'에포크 : {epoch} , 손실(오차) : {loss.item()}')

# 모델 평가
# model.train() ; 학습모드
# model.eval()  : 평가/추론모드 
model.eval()
with torch.no_grad() :
    train_pred = model(X_train)
    test_pred = model(X_test)

from sklearn.metrics import r2_score
print(r2_score(Y_train.numpy() , train_pred.numpy()))
print(r2_score(Y_test.numpy() , test_pred.numpy()))

# 모델 추론
new_students = [ [4, 0, 16, 17],[10,60,89,200], [1, 10, 7, 6] ]
new_train = torch.tensor( ss.transform( new_students ), dtype=torch.float32 ) 
with torch.no_grad() : 
  new_scores = model( new_train )

for i in range(len(new_students)) :
  print( f'학생{i+1} 예측 최종성적(G3) : {new_scores[i].item()}' ) 


# 10000번 반복
# 0.7721379399299622
# 0.5998789072036743
# 학생1 예측 최종성적(G3) :  14.373217582702637
# 학생2 예측 최종성적(G3) :  5.709449768066406

# 20000번 반복
# 0.8991169929504395
# 0.7515567541122437
# 학생1 예측 최종성적(G3) :  14.979403495788574
# 학생2 예측 최종성적(G3) :  4.328333854675293

